# YOLO11s 베이스라인 학습

**전제:** 팀원 노트북(`pill_detection_dataset.ipynb`)을 실행해 `data/processed` 아래에
`images/{train,val,test}`, `labels/{train,val,test}`, `data.yaml` 이 생성돼 있어야 합니다.
이 노트북(`notebooks/` 안에 위치)은 그 `data.yaml`만 받아 YOLO11s를 학습합니다.

베이스라인은 팀 합의대로 **기본값 위주**로 돌립니다.


## 1. 설치 환경 재현성

In [ ]:
# ============================================================
# 1. 설치, 환경, 재현성
# ============================================================
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("PILL_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "requirements.txt").is_file(), (
    "저장소 루트에서 노트북을 실행하거나 PILL_PROJECT_ROOT를 지정하세요: "
    f"{PROJECT_ROOT}"
)

%pip install -q -r {PROJECT_ROOT / "requirements.txt"}

import random
import numpy as np
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("project:", PROJECT_ROOT)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| device:", DEVICE)


## 2. 경로 설정

In [ ]:
# Colab에서 Google Drive 데이터가 필요할 때만 마운트합니다.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


In [ ]:
# ============================================================
# 2-2. 경로 설정 (PROJECT_ROOT 기준 상대경로)
# ============================================================
DATASET_DIR = PROJECT_ROOT / "data" / "processed"
TEST_IMAGE_DIR = PROJECT_ROOT / "data" / "dataset" / "cleaning_data" / "test_images"
ANNOTATION_DIR = PROJECT_ROOT / "data" / "dataset" / "cleaning_data" / "train_annotations"
WEIGHTS_PATH = PROJECT_ROOT / "outputs" / "checkpoints" / "yolo11s_best.pt"
SUBMISSION_PATH = PROJECT_ROOT / "outputs" / "submission" / "submission.csv"

print("DATASET_DIR    exists:", DATASET_DIR.exists(), "(필수)")
print("TEST_IMAGE_DIR exists:", TEST_IMAGE_DIR.exists(), "(필수)")
print("ANNOTATION_DIR exists:", ANNOTATION_DIR.exists(), "(필수)")
print("WEIGHTS_PATH   exists:", WEIGHTS_PATH.exists(), "(학습 후 생성)")
print("SUBMISSION dir:", SUBMISSION_PATH.parent)


## 3. data.yaml

In [ ]:
# ==================================================================
# 3. data.yaml 경로 이식성 처리
# ==================================================================

import yaml
src_yaml = DATASET_DIR / "data.yaml"
cfg = yaml.safe_load(open(src_yaml, encoding="utf-8"))
cfg["path"] = str(DATASET_DIR.resolve())
yaml.safe_dump(cfg, open(src_yaml, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
RUNTIME_YAML = src_yaml          # 이후 코드는 그대로 RUNTIME_YAML 사용

assert (DATASET_DIR / "images" / "train").is_dir(), \
    "images/train 이 없습니다. pill_detection_dataset.ipynb 를 먼저 실행해 data/processed 를 만드세요."

print("클래스 수:", cfg["nc"], "| yaml:", RUNTIME_YAML)

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.yolo.yolo_mapping import build_yolo_to_category_id
yolo_to_category_id = build_yolo_to_category_id(ANNOTATION_DIR)
assert len(yolo_to_category_id) == int(cfg["nc"]), (
    f"매핑 {len(yolo_to_category_id)} != data.yaml {cfg['nc']}"
)


## 4. 학습 (W&B 로깅 포함)

In [ ]:
# ==================================================================
# 4-1. W&B 설치 및 로그인
# ==================================================================
import wandb
from ultralytics import settings

settings.update({"wandb": True})          # Ultralytics 내장 W&B 로깅 ON
wandb.login()   # 최초 1회, wandb.ai/authorize 의 API 키


# YOLO11s + YOLO11m WBF ensemble

기존 데이터 구조, split, class mapping, augmentation, submission schema를 유지합니다.
`TRAIN_YOLO11M=False`이면 기존 weight만 로드하므로 YOLO11s를 다시 학습하지 않습니다.


## 1. Configuration


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
IMGSZ = 960
YOLO11S_BATCH = 16
YOLO11M_BATCH = 8                 # OOM이면 4 또는 -1(자동)로 변경
TRAIN_YOLO11S = False
TRAIN_YOLO11M = False             # 기존 best.pt 사용 시 False

YOLO11S_WEIGHTS = WEIGHTS_PATH    # 기존 최고 YOLO11s best.pt (절대 덮어쓰지 않음)
YOLO11M_EXPERIMENT = "yolo11m_mix8_img960"
YOLO11M_RUN_DIR = PROJECT_ROOT / "outputs" / "yolo" / YOLO11M_EXPERIMENT
YOLO11M_WEIGHTS = PROJECT_ROOT / "outputs" / "checkpoints" / "yolo11m_best.pt"

PRED_CONF = 0.05                  # WBF 전에 후보를 충분히 보존
WBF_IOU_THR = 0.55
WBF_SKIP_BOX_THR = 0.05
WBF_OUTPUT_CONF = 0.05            # WBF 후 최종 confidence filter
MODEL_WEIGHTS = [1.0, 1.0]        # 예: [1.0, 1.2], [1.0, 1.5]
MAX_DET = 4                       # 초과할 때만 confidence 상위 4개

WBF_SUBMISSION_PATH = SUBMISSION_PATH.parent / "submission_yolo11s_yolo11m_wbf.csv"
YOLO11S_SUBMISSION_PATH = SUBMISSION_PATH.parent / "submission_yolo11s.csv"
YOLO11M_SUBMISSION_PATH = SUBMISSION_PATH.parent / "submission_yolo11m.csv"

print("YOLO11S_WEIGHTS:", YOLO11S_WEIGHTS, "| exists:", YOLO11S_WEIGHTS.exists())
print("YOLO11M_WEIGHTS:", YOLO11M_WEIGHTS, "| exists:", YOLO11M_WEIGHTS.exists())


## Optional YOLO11s training (기존 최고 설정 보존)

기본은 실행하지 않습니다. `TRAIN_YOLO11S=True`일 때만 기존 YOLO11s 설정으로 학습하며, 기존 `YOLO11S_WEIGHTS` 경로를 자동으로 바꾸거나 덮어쓰지 않습니다.


In [ ]:
# ============================================================
# Optional YOLO11s training: 기존 최고 설정 (imgsz=960)
# ============================================================
if TRAIN_YOLO11S:
    yolo11s_train_model = YOLO("yolo11s.pt")
    yolo11s_train_results = yolo11s_train_model.train(
        data=str(RUNTIME_YAML), epochs=50, patience=10,
        imgsz=IMGSZ, batch=YOLO11S_BATCH,
        seed=SEED, deterministic=True, device=DEVICE, workers=2,
        project=str(PROJECT_ROOT / "outputs" / "yolo"),
        name="yolo11s_mix8_img960_retrain", exist_ok=False,
        hsv_h=0.015, hsv_s=0.3, hsv_v=0.4,
        degrees=90.0, translate=0.1, scale=0.0, shear=0.5, perspective=0.0,
        flipud=0.0, fliplr=0.0, mosaic=0.0, mixup=0.0,
        copy_paste=0.0, erasing=0.0,
    )
    print("새 YOLO11s 결과(기존 weight와 별도):", yolo11s_train_results.save_dir)


## 2. YOLO11s weight 확인

기존 YOLO11s는 재학습하지 않고 지정된 `best.pt`를 직접 로드합니다.


In [ ]:
# ============================================================
# 2. YOLO11s weight 확인/로드
# ============================================================
assert YOLO11S_WEIGHTS.exists(), f"YOLO11s best.pt가 없습니다: {YOLO11S_WEIGHTS}"
yolo11s_model = YOLO(str(YOLO11S_WEIGHTS))
print("기존 YOLO11s 로드 완료:", YOLO11S_WEIGHTS)


## 3. YOLO11m training

YOLO11s와 같은 `RUNTIME_YAML`(동일 train/val split), epoch, patience, augmentation을 사용합니다. 결과 폴더는 별도 이름을 사용합니다.


In [ ]:
# ============================================================
# 3. YOLO11m training (기존 YOLO11s 설정 유지, batch만 변수화)
# ============================================================
if TRAIN_YOLO11M:
    yolo11m_train_model = YOLO("yolo11m.pt")
    yolo11m_results = yolo11m_train_model.train(
        data=str(RUNTIME_YAML), epochs=50, patience=10,
        imgsz=IMGSZ, batch=YOLO11M_BATCH,
        seed=SEED, deterministic=True, device=DEVICE, workers=2,
        project=str(PROJECT_ROOT / "outputs" / "yolo"),
        name=YOLO11M_EXPERIMENT, exist_ok=False,
        hsv_h=0.015, hsv_s=0.3, hsv_v=0.4,
        degrees=90.0, translate=0.1, scale=0.0, shear=0.5, perspective=0.0,
        flipud=0.0, fliplr=0.0, mosaic=0.0, mixup=0.0,
        copy_paste=0.0, erasing=0.0,
    )
    YOLO11M_WEIGHTS = Path(yolo11m_results.save_dir) / "weights" / "best.pt"
    print("YOLO11m 학습 완료:", YOLO11M_WEIGHTS)
else:
    print("YOLO11m 학습 생략. 기존 weight를 사용합니다:", YOLO11M_WEIGHTS)

assert YOLO11M_WEIGHTS.exists(), (
    f"YOLO11m best.pt가 없습니다: {YOLO11M_WEIGHTS}\n"
    "처음 학습할 때 TRAIN_YOLO11M=True로 변경하세요."
)
yolo11m_model = YOLO(str(YOLO11M_WEIGHTS))


## 4–5. Validation — YOLO11s / YOLO11m


In [ ]:
# ============================================================
# 4-5. 같은 validation set에서 단독 모델 비교
# ============================================================
import pandas as pd

def validate_yolo(model, model_name):
    metrics = model.val(
        data=str(RUNTIME_YAML), split="val", imgsz=IMGSZ,
        device=DEVICE, verbose=False,
        project=str(PROJECT_ROOT / "outputs" / "yolo"),
        name=f"val_{model_name}_img{IMGSZ}", exist_ok=True,
    )
    row = {
        "model": model_name,
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    }
    names = metrics.names if hasattr(metrics, "names") else model.names
    class_ap = pd.DataFrame({
        "class_id": list(range(len(metrics.box.maps))),
        "class_name": [names[i] for i in range(len(metrics.box.maps))],
        "AP50-95": [float(x) for x in metrics.box.maps],
    })
    return metrics, row, class_ap

s_val_metrics, s_val_row, s_class_ap = validate_yolo(yolo11s_model, "YOLO11s")
m_val_metrics, m_val_row, m_class_ap = validate_yolo(yolo11m_model, "YOLO11m")

single_model_table = pd.DataFrame([s_val_row, m_val_row]).set_index("model")
display(single_model_table.style.format("{:.4f}"))
print("YOLO11s class별 AP")
display(s_class_ap)
print("YOLO11m class별 AP")
display(m_class_ap)


## 6. WBF utility functions

`ensemble-boxes`의 `weighted_boxes_fusion`을 사용합니다. 입력은 normalized `xyxy`, 출력은 원본 pixel `xyxy`입니다.


In [ ]:
# ensemble-boxes는 requirements.txt에서 설치됩니다.


In [ ]:
# ============================================================
# 6-2. prediction / WBF utilities
# detection 배열 형식: [x1, y1, x2, y2, confidence, yolo_class_id]
# ============================================================
from collections import Counter
from ensemble_boxes import weighted_boxes_fusion
from PIL import Image
import numpy as np
import torch
import gc

def _result_to_numpy(result):
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 6), dtype=np.float32)
    xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)[:, None]
    cls = result.boxes.cls.detach().cpu().numpy().astype(np.float32)[:, None]
    return np.concatenate([xyxy, conf, cls], axis=1)

def predict_single_model(model, image_paths, pred_conf=PRED_CONF):
    """입력 순서를 보존해 {Path: pixel-xyxy detection array}를 반환합니다."""
    output = {}
    for i, path in enumerate(image_paths):
        result = model.predict(
            str(path), imgsz=IMGSZ, conf=pred_conf, device=DEVICE,
            max_det=300, verbose=False,
        )[0]
        output[Path(path)] = _result_to_numpy(result)
        del result
        if i % 100 == 0:
            gc.collect()
            print(f"{i}/{len(image_paths)} prediction")
    return output

def fuse_image_detections(model_detections, width, height,
                          model_weights=MODEL_WEIGHTS,
                          iou_thr=WBF_IOU_THR,
                          skip_box_thr=WBF_SKIP_BOX_THR,
                          output_conf=WBF_OUTPUT_CONF,
                          max_det=MAX_DET):
    """모델별 pixel xyxy를 normalize해 WBF 후 pixel xyxy로 정확히 복원합니다."""
    boxes_list, scores_list, labels_list = [], [], []
    for det in model_detections:
        det = np.asarray(det, dtype=np.float32).reshape(-1, 6)
        if len(det) == 0:
            boxes_list.append([]); scores_list.append([]); labels_list.append([])
            continue
        boxes = det[:, :4].copy()
        boxes[:, [0, 2]] /= float(width)   # x / width
        boxes[:, [1, 3]] /= float(height) # y / height
        boxes = np.clip(boxes, 0.0, 1.0)
        valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
        boxes_list.append(boxes[valid].tolist())
        scores_list.append(det[valid, 4].tolist())
        labels_list.append(det[valid, 5].astype(int).tolist())

    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=list(model_weights), iou_thr=float(iou_thr),
        skip_box_thr=float(skip_box_thr), conf_type="avg",
    )
    if len(boxes) == 0:
        return np.empty((0, 6), dtype=np.float32), 0

    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.float32)
    boxes[:, [0, 2]] *= float(width)
    boxes[:, [1, 3]] *= float(height)
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, width)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, height)

    keep = scores >= float(output_conf)
    fused = np.column_stack([boxes[keep], scores[keep], labels[keep]]).astype(np.float32)
    fused = fused[np.argsort(-fused[:, 4])] if len(fused) else fused
    count_before_max_det = len(fused)
    if max_det is not None and len(fused) > max_det:
        fused = fused[:max_det]            # 4개 초과일 때만 자름
    return fused, count_before_max_det

def predict_wbf(image_paths, s_predictions, m_predictions, **wbf_kwargs):
    fused_predictions, before_max_counts = {}, {}
    for path in image_paths:
        path = Path(path)
        with Image.open(path) as image:
            width, height = image.size     # PIL: (width, height)
        fused, before_count = fuse_image_detections(
            [s_predictions[path], m_predictions[path]], width, height, **wbf_kwargs
        )
        fused_predictions[path] = fused
        before_max_counts[path] = before_count
    return fused_predictions, before_max_counts


## 7. WBF validation

아래 평가는 validation label을 직접 읽어 WBF 결과의 AP를 계산합니다. precision/recall은 IoU 0.5 및 현재 `WBF_OUTPUT_CONF` 기준입니다. 여러 설정을 `evaluate_wbf_grid`로 비교할 수 있습니다.


In [ ]:
# ============================================================
# 7-1. WBF validation metric utilities
# ============================================================
def box_iou_numpy(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)
    lt = np.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = np.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    a1 = np.prod(boxes1[:, 2:] - boxes1[:, :2], axis=1)[:, None]
    a2 = np.prod(boxes2[:, 2:] - boxes2[:, :2], axis=1)[None, :]
    return inter / np.clip(a1 + a2 - inter, 1e-9, None)

def load_yolo_ground_truth(image_path):
    label_path = DATASET_DIR / "labels" / "val" / f"{image_path.stem}.txt"
    with Image.open(image_path) as image:
        width, height = image.size
    if not label_path.exists() or not label_path.read_text().strip():
        return np.empty((0, 5), dtype=np.float32)
    y = np.loadtxt(label_path, ndmin=2, dtype=np.float32)
    cls, cx, cy, bw, bh = y[:, 0], y[:, 1], y[:, 2], y[:, 3], y[:, 4]
    xyxy = np.column_stack([
        (cx - bw / 2) * width, (cy - bh / 2) * height,
        (cx + bw / 2) * width, (cy + bh / 2) * height,
    ])
    return np.column_stack([xyxy, cls]).astype(np.float32)

def _average_precision(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(np.trapz(np.interp(x, mrec, mpre), x))

def evaluate_detection_dict(predictions, image_paths, num_classes):
    iou_thresholds = np.arange(0.50, 0.96, 0.05)
    gt_count = np.zeros(num_classes, dtype=int)
    records = {(c, t): [] for c in range(num_classes) for t in range(len(iou_thresholds))}

    for path in image_paths:
        gt = load_yolo_ground_truth(Path(path))
        pred = np.asarray(predictions[Path(path)], dtype=np.float32).reshape(-1, 6)
        for c in range(num_classes):
            gt_c = gt[gt[:, 4].astype(int) == c, :4]
            pred_c = pred[pred[:, 5].astype(int) == c]
            gt_count[c] += len(gt_c)
            pred_c = pred_c[np.argsort(-pred_c[:, 4])] if len(pred_c) else pred_c
            ious = box_iou_numpy(pred_c[:, :4], gt_c)
            for ti, thr in enumerate(iou_thresholds):
                used = set()
                for pi, p in enumerate(pred_c):
                    best = int(np.argmax(ious[pi])) if len(gt_c) else -1
                    is_tp = best >= 0 and ious[pi, best] >= thr and best not in used
                    if is_tp:
                        used.add(best)
                    records[(c, ti)].append((float(p[4]), int(is_tp)))

    ap = np.full((num_classes, len(iou_thresholds)), np.nan, dtype=float)
    total_tp50 = total_fp50 = 0
    for c in range(num_classes):
        for ti in range(len(iou_thresholds)):
            recs = sorted(records[(c, ti)], reverse=True)
            if gt_count[c] == 0:
                continue
            if not recs:
                ap[c, ti] = 0.0
                continue
            tp = np.array([x[1] for x in recs], dtype=float)
            fp = 1.0 - tp
            tp_cum, fp_cum = np.cumsum(tp), np.cumsum(fp)
            recall = tp_cum / gt_count[c]
            precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
            ap[c, ti] = _average_precision(recall, precision)
            if ti == 0:
                total_tp50 += int(tp.sum()); total_fp50 += int(fp.sum())

    valid = gt_count > 0
    total_gt = int(gt_count.sum())
    summary = {
        "model": "YOLO11s + YOLO11m WBF",
        "mAP50": float(np.nanmean(ap[valid, 0])),
        "mAP50-95": float(np.nanmean(ap[valid])),
        "precision": total_tp50 / max(total_tp50 + total_fp50, 1),
        "recall": total_tp50 / max(total_gt, 1),
    }
    return summary, ap

def evaluate_wbf(image_paths, s_predictions, m_predictions,
                 iou_thr=WBF_IOU_THR, skip_box_thr=WBF_SKIP_BOX_THR,
                 model_weights=MODEL_WEIGHTS, output_conf=WBF_OUTPUT_CONF,
                 max_det=MAX_DET):
    fused, before_counts = predict_wbf(
        image_paths, s_predictions, m_predictions,
        iou_thr=iou_thr, skip_box_thr=skip_box_thr,
        model_weights=model_weights, output_conf=output_conf, max_det=max_det,
    )
    summary, class_ap = evaluate_detection_dict(fused, image_paths, int(cfg["nc"]))
    summary.update({
        "WBF_IOU_THR": iou_thr,
        "WBF_SKIP_BOX_THR": skip_box_thr,
        "MODEL_WEIGHTS": str(list(model_weights)),
    })
    return summary, class_ap, fused, before_counts

def evaluate_wbf_grid(settings, image_paths, s_predictions, m_predictions):
    rows = []
    for params in settings:
        summary, _, _, _ = evaluate_wbf(
            image_paths, s_predictions, m_predictions, **params
        )
        rows.append(summary)
    return pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)


In [ ]:
# ============================================================
# 7-2. validation prediction / WBF 평가
# ============================================================
val_files = sorted(
    (DATASET_DIR / "images" / "val").glob("*.*"),
    key=lambda p: p.name,
)
val_files = [p for p in val_files if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
print("validation images:", len(val_files))

s_val_predictions = predict_single_model(yolo11s_model, val_files)
m_val_predictions = predict_single_model(yolo11m_model, val_files)

wbf_summary, wbf_class_ap, wbf_val_predictions, wbf_val_before_counts = evaluate_wbf(
    val_files, s_val_predictions, m_val_predictions,
    iou_thr=WBF_IOU_THR,
    skip_box_thr=WBF_SKIP_BOX_THR,
    model_weights=MODEL_WEIGHTS,
)

comparison_table = pd.concat([
    single_model_table.reset_index(),
    pd.DataFrame([{k: wbf_summary[k] for k in single_model_table.reset_index().columns}]),
], ignore_index=True).set_index("model")
display(comparison_table.style.format("{:.4f}"))

wbf_class_table = pd.DataFrame({
    "class_id": np.arange(int(cfg["nc"])),
    "class_name": [cfg["names"][i] if isinstance(cfg["names"], list) else cfg["names"].get(i, cfg["names"].get(str(i), str(i))) for i in range(int(cfg["nc"]))],
    "AP50": wbf_class_ap[:, 0],
    "AP50-95": np.nanmean(wbf_class_ap, axis=1),
})
display(wbf_class_table)

# 여러 설정 비교 예시 (필요한 조합을 추가/수정)
WBF_GRID = [
    {"iou_thr": 0.55, "skip_box_thr": 0.05, "model_weights": [1.0, 1.0]},
    # {"iou_thr": 0.50, "skip_box_thr": 0.05, "model_weights": [1.0, 1.2]},
    # {"iou_thr": 0.60, "skip_box_thr": 0.03, "model_weights": [1.0, 1.5]},
]
display(evaluate_wbf_grid(WBF_GRID, val_files, s_val_predictions, m_val_predictions))


## 8–9. Test inference + WBF


In [ ]:
# ============================================================
# 8-9. test inference: YOLO11s + YOLO11m -> WBF
# 숫자형 image_id 순서를 기존 submission 코드와 동일하게 유지
# ============================================================
test_files = sorted(TEST_IMAGE_DIR.glob("*.png"), key=lambda p: int(p.stem))
assert test_files, f"test image가 없습니다: {TEST_IMAGE_DIR}"

s_test_predictions = predict_single_model(yolo11s_model, test_files)
m_test_predictions = predict_single_model(yolo11m_model, test_files)
wbf_test_predictions, wbf_test_before_counts = predict_wbf(
    test_files, s_test_predictions, m_test_predictions,
    iou_thr=WBF_IOU_THR,
    skip_box_thr=WBF_SKIP_BOX_THR,
    model_weights=MODEL_WEIGHTS,
    output_conf=WBF_OUTPUT_CONF,
    max_det=MAX_DET,
)


## 10. Kaggle submission 생성

기존 CSV의 column, annotation/image/category ID, pixel `xywh`, 반올림 및 저장 방식을 그대로 사용합니다. 기본 출력은 WBF이며 같은 함수로 단독 모델 A/B 파일도 만들 수 있습니다.


In [ ]:
# ============================================================
# 10. 기존 Kaggle 포맷 그대로 저장
# ============================================================
import csv

SUBMISSION_COLUMNS = [
    "annotation_id", "image_id", "category_id",
    "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score",
]

def create_submission(predictions, image_files, output_path):
    rows, ann_id = [], 1
    for f in image_files:
        image_id = int(f.stem)
        det = np.asarray(predictions[Path(f)], dtype=np.float32).reshape(-1, 6)
        for x1, y1, x2, y2, conf, cls in det:
            rows.append([
                ann_id, image_id, yolo_to_category_id[int(cls)],
                int(round(float(x1))), int(round(float(y1))),
                int(round(float(x2 - x1))), int(round(float(y2 - y1))),
                round(float(conf), 4),
            ])
            ann_id += 1
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(SUBMISSION_COLUMNS)
        writer.writerows(rows)
    print("작성 완료:", output_path, "| 행:", len(rows))
    return pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)

# C. 기본 제출: YOLO11s + YOLO11m WBF
submission_df = create_submission(wbf_test_predictions, test_files, WBF_SUBMISSION_PATH)

# A/B 단독 제출이 필요하면 아래 주석 해제. MAX_DET 적용을 원하면 별도 후처리 후 전달하세요.
# create_submission(s_test_predictions, test_files, YOLO11S_SUBMISSION_PATH)
# create_submission(m_test_predictions, test_files, YOLO11M_SUBMISSION_PATH)


## 11. Sanity check


In [ ]:
# ============================================================
# 11. 첫 이미지 상세 + 전체 test 통계 + CSV schema 검증
# ============================================================
first = test_files[0]
print("첫 이미지:", first.name)
print("YOLO11s detection 개수:", len(s_test_predictions[first]))
print("YOLO11m detection 개수:", len(m_test_predictions[first]))
print("WBF 후 detection 개수:", len(wbf_test_predictions[first]))
for j, (x1, y1, x2, y2, conf, cls) in enumerate(wbf_test_predictions[first], 1):
    cls_int = int(cls)
    print({
        "detection": j,
        "yolo_class": cls_int,
        "confidence": round(float(conf), 6),
        "bbox_xyxy_pixel": [round(float(v), 2) for v in (x1, y1, x2, y2)],
        "kaggle_category_id": yolo_to_category_id[cls_int],
    })

counts = np.array([len(wbf_test_predictions[p]) for p in test_files], dtype=int)
before_counts = np.array([wbf_test_before_counts[p] for p in test_files], dtype=int)
print("총 이미지 수:", len(test_files))
print("총 detection 수:", int(counts.sum()))
print("이미지당 평균 detection 수:", round(float(counts.mean()), 4))
for n in range(5):
    print(f"detection {n}개인 이미지 수:", int((counts == n).sum()))
print("WBF 직후(MAX_DET 전) 4개 초과 이미지 수:", int((before_counts > 4).sum()))
print("WBF 직후(MAX_DET 전) 4개 초과 detection 수:", int(np.maximum(before_counts - 4, 0).sum()))

saved_df = pd.read_csv(WBF_SUBMISSION_PATH)
assert saved_df.columns.tolist() == SUBMISSION_COLUMNS
assert saved_df["annotation_id"].is_unique
assert (saved_df[["bbox_w", "bbox_h"]] > 0).all().all()
assert saved_df["score"].between(0, 1).all()
print("submission shape:", saved_df.shape)
print("columns:", saved_df.columns.tolist())
display(saved_df.head())


In [ ]:
# ============================================================
# 9-2. eval 이미지: 정답 vs 예측 나란히 시각화
# ============================================================
# 한글 폰트 (Colab)
!apt-get -qq install -y fonts-nanum >/dev/null 2>&1
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
from matplotlib.patches import Rectangle
from PIL import Image
for fp in fm.findSystemFonts():
    if "Nanum" in fp: fm.fontManager.addfont(fp)
plt.rcParams["font.family"] = "NanumGothic"; plt.rcParams["axes.unicode_minus"] = False

names = yolo11s_model.names   # {0: '보령부스파정 5mg', ...}
img_dir = DATASET_DIR / "images" / "test"
lbl_dir = DATASET_DIR / "labels" / "test"

def gt_boxes(stem, W, H):
    p = lbl_dir / f"{stem}.txt"
    out = []
    if p.exists():
        for line in p.read_text().strip().splitlines():
            c, cx, cy, w, h = map(float, line.split())
            out.append((int(c), (cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H))
    return out

eval_imgs = sorted(img_dir.glob("*.png"))[:4]
fig, axes = plt.subplots(len(eval_imgs), 2, figsize=(12, 5*len(eval_imgs)))
for row, ip in enumerate(eval_imgs):
    img = Image.open(ip).convert("RGB"); W, H = img.size
    # 정답
    ax = axes[row, 0]; ax.imshow(img); ax.set_title(f"정답 GT: {ip.name}"); ax.axis("off")
    for c, x1, y1, x2, y2 in gt_boxes(ip.stem, W, H):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", lw=2))
        ax.text(x1, y1-5, names[c], color="lime", fontsize=8)
    # 예측
    r = yolo11s_model.predict(str(ip), imgsz=640, conf=0.05, max_det=4, device=DEVICE, verbose=False)[0]
    ax = axes[row, 1]; ax.imshow(img); ax.set_title("예측 Pred"); ax.axis("off")
    for (x1,y1,x2,y2), c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", lw=2))
        ax.text(x1, y1-5, f"{names[int(c)]} {s:.2f}", color="red", fontsize=8)
plt.tight_layout(); plt.show()


## 10. submission.csv 예측 결과물 시각화

(아래 코드는 민협님 작성 노트북 파일인 pill_detection_dataset.ipynb 의 마지막 부분을 가져와 수정한 것)

이미지 842장 기준 8분 정도 소요

이미지를 영구보관 하거나 / 드라이브에 폴더를 만들어 다운받고 싶은 경우,  

`# output_dir = PROJECT_ROOT / "outputs" / "pred_visualizations"`

 을 uncomment 하면 됨

In [ ]:
from pathlib import Path
import math, json
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image
import pandas as pd
from urllib.request import Request, urlopen
import tempfile, shutil

# ============================================================
# 1. 한글 폰트 설정
# ============================================================
FONT_URL = ("https://raw.githubusercontent.com/google/fonts/"
            "main/ofl/nanumgothic/NanumGothic-Regular.ttf")
FONT_DIR = Path.home() / ".cache" / "matplotlib-korean-font"
FONT_PATH = FONT_DIR / "NanumGothic-Regular.ttf"

def download_font_if_needed():
    if FONT_PATH.exists() and FONT_PATH.stat().st_size > 0:
        return FONT_PATH
    FONT_DIR.mkdir(parents=True, exist_ok=True)
    req = Request(FONT_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(req, timeout=30) as resp:
        with tempfile.NamedTemporaryFile(dir=FONT_DIR, suffix=".tmp", delete=False) as tf:
            shutil.copyfileobj(resp, tf); tmp = Path(tf.name)
    tmp.replace(FONT_PATH); return FONT_PATH

try:
    fp = download_font_if_needed()
    fm.fontManager.addfont(str(fp))
    font_property = fm.FontProperties(fname=str(fp))
    plt.rcParams["font.family"] = font_property.get_name()
    plt.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 준비 실패:", e); font_property = None

# ============================================================
# 2. 시각화 설정
# ============================================================
images_per_page, num_cols = 10, 5
num_rows = math.ceil(images_per_page / num_cols)
save_figures, show_figures = True, False
# output_dir = PROJECT_ROOT / "outputs" / "pred_visualizations"
output_dir = PROJECT_ROOT / "outputs" / "predictions" / "pred_visualizations"
output_dir.mkdir(parents=True, exist_ok=True)
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# ============================================================
# 3. category_id → 약 이름 매핑
# ============================================================
cat2name = {}
for jf in ANNOTATION_DIR.rglob("*.json"):
    for c in json.load(open(jf, encoding="utf-8")).get("categories", []):
        cat2name[int(c["id"])] = c["name"]

def find_image(image_id):                       # 파일명 형식 자동 탐색
    for pat in (f"{image_id}.png", f"{image_id}.jpg", f"image{image_id}.png"):
        p = TEST_IMAGE_DIR / pat
        if p.exists(): return p
    return TEST_IMAGE_DIR / f"{image_id}.png"

# ============================================================
# 4. submission.csv 예측 시각화  (★ image_id 단위로 그룹)
# ============================================================
sub = pd.read_csv(SUBMISSION_PATH)
image_ids = sorted(sub["image_id"].unique())    # ★ annotation_id 아님! image_id로 묶음
# image_ids = image_ids[:30]                    # ← 먼저 일부만 미리보려면 주석 해제

num_images = len(image_ids)
num_pages = math.ceil(num_images / images_per_page)
print(f"전체 이미지 수: {num_images}\n생성할 페이지 수: {num_pages}")

for page_index in range(num_pages):
    start, end = page_index*images_per_page, min((page_index+1)*images_per_page, num_images)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(25, 11), constrained_layout=True)
    axes = np.asarray(axes).reshape(-1)

    for sub_i, image_id in enumerate(image_ids[start:end]):
        ax = axes[sub_i]
        img_path = find_image(image_id)
        try:
            ax.imshow(Image.open(img_path).convert("RGB"))
        except FileNotFoundError:
            ax.text(0.5, 0.5, f"{img_path.name}\n없음", ha="center"); ax.axis("off"); continue

        dets = sub[sub["image_id"] == image_id]          # ★ 그 이미지의 모든 검출(행)
        for j, (_, r) in enumerate(dets.iterrows()):
            x, y, w, h = float(r["bbox_x"]), float(r["bbox_y"]), float(r["bbox_w"]), float(r["bbox_h"])
            color = colors[j % len(colors)]
            ax.add_patch(Rectangle((x, y), w, h, linewidth=2.5, edgecolor=color, facecolor="none"))
            name = cat2name.get(int(r["category_id"]), "unknown")
            text = f"{name}\nID: {int(r['category_id'])}\nscore: {r['score']:.2f}"
            ty, va = (y - 5, "bottom") if y >= 70 else (y + 5, "top")
            ax.text(x, ty, text, fontsize=8, color="white", verticalalignment=va,
                    fontproperties=font_property,
                    bbox={"facecolor": color, "alpha": 0.85, "edgecolor": "none", "pad": 2})

        ax.set_title(f"[{image_id}] {img_path.name}\n검출 수: {len(dets)}",
                     fontsize=9, fontproperties=font_property)
        ax.axis("off")

    for k in range(end - start, len(axes)):
        axes[k].axis("off")

    fig.suptitle(f"YOLO 예측 시각화 ({start+1}–{end} / {num_images})",
                 fontsize=18, fontproperties=font_property)

    if save_figures:
        out = output_dir / f"pred_page_{page_index+1:03d}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight", facecolor="white")
        if (page_index+1) % 10 == 0 or page_index+1 == num_pages:
            print(f"[{page_index+1:03d}/{num_pages:03d}] 저장 완료")
    if show_figures:
        plt.show()
    plt.close(fig)

print(f"\n전체 시각화 완료: {output_dir.resolve()}")


In [ ]:
from IPython.display import Image as IPyImage, display
from pathlib import Path

pages = sorted(PROJECT_ROOT / "outputs" / "predictions" / "pred_visualizations".glob("*.png"))
print(f"{len(pages)}장 표시")
for p in pages:                 # 일부만 보려면 pages[:10]
    display(IPyImage(filename=str(p), width=1400))
